<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# MarketPulse — Analytics Marketing Digital
## Notebook 4 — Dashboard Power BI · Vue Account Manager
### ✅ VERSION CORRIGÉE

> **Comment lire ce guide :**  
> Les blocs `MÉTHODE` expliquent les choix de conception du dashboard.  
> Les blocs `DAX` contiennent les formules prêtes à copier-coller dans Power BI.  
> Les blocs `CONFIGURATION` décrivent les visuels et leurs paramètres.

| | |
|---|---|
| **Prérequis** | NB1, NB2, NB3 complétés |
| **Outil** | Power BI Desktop |
| **Durée estimée** | 4h à 5h |

> **Objectif :** Construire un dashboard 5 pages exploitable **quotidiennement** par les account managers de MarketPulse. Chaque page répond à une question opérationnelle précise. La page 5 (Alertes ML) intègre directement le fichier de scores produit par le NB5 — `campagnes_risque_scores.csv` — pour afficher les campagnes en dérive dans un bandeau rouge conditionnel.

> **Fichiers à connecter dans Power BI :**
> - `marketpulse_analytics.csv` — table centrale (354 lignes, granularité campagne)
> - `performances_daily.csv` — table des faits quotidiens (8 861 lignes)
> - `clients_agence.csv` — référentiel clients (25 lignes)
> - `audiences.csv` — référentiel audiences (80 lignes)
> - `contacts_crm.csv` — LTV et rétention (9 500 lignes)
> - `campagnes_risque_scores.csv` — sortie NB5, scores ML (354 lignes)

---
## 0. Préparation des fichiers et vérification des données

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/marketpulse/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)

BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/marketpulse/data/'

# Chargement tables
analytics_path = f'{SAVE_PATH}marketpulse_analytics.csv'
if os.path.exists(analytics_path):
    df_an = pd.read_csv(analytics_path, parse_dates=['date_debut','date_fin'])
    print('marketpulse_analytics chargé depuis SAVE_PATH')
else:
    df_an = pd.read_csv(BASE_URL + 'marketpulse_analytics.csv', parse_dates=['date_debut','date_fin'])
    print('marketpulse_analytics chargé depuis GitHub')

df_perf = pd.read_csv(BASE_URL + 'performances_daily.csv', parse_dates=['date_perf'])
df_cli  = pd.read_csv(BASE_URL + 'clients_agence.csv')
df_aud  = pd.read_csv(BASE_URL + 'audiences.csv')
df_cont = pd.read_csv(BASE_URL + 'contacts_crm.csv')

print('\nTables disponibles :')
for nom, df in [('marketpulse_analytics',df_an),('performances_daily',df_perf),
                 ('clients_agence',df_cli),('audiences',df_aud),('contacts_crm',df_cont)]:
    print(f'  {nom:<30} {len(df):>8,} lignes × {df.shape[1]} colonnes')

In [ ]:
# Vérification des KPIs de référence avant construction du dashboard
df_clean = df_perf[
    (df_perf['clics'] <= df_perf['impressions']) &
    (df_perf['revenu_genere_fcfa'] >= 0) &
    (df_perf['roas'] <= 50)
].copy()

df_clean['ctr_r']  = df_clean['clics'] / df_clean['impressions'].replace(0, np.nan)
df_clean['roas_r'] = df_clean['revenu_genere_fcfa'] / df_clean['depenses_fcfa'].replace(0, np.nan)
df_clean = df_clean[df_clean['ctr_r'] <= 0.20]

print('=== KPIs DE RÉFÉRENCE (valeurs attendues dans Power BI) ===')
print(f'  ROAS global moyen          : {df_clean["roas_r"].mean():.3f}×')
print(f'  CTR global moyen           : {df_clean["ctr_r"].mean()*100:.2f}%')
print(f'  CPC moyen (FCFA)           : {(df_clean["depenses_fcfa"]/df_clean["clics"].replace(0,np.nan)).mean():.0f}')
print(f'  Nb campagnes               : {len(df_an)}')
print(f'  Campagnes sous-perf        : {df_an["campagne_sous_performe"].sum()} ({df_an["campagne_sous_performe"].mean()*100:.1f}%)')
print(f'\nRépartition par canal :')
for canal, grp in df_clean.groupby('canal'):
    print(f'  {canal:<12} ROAS={grp["roas_r"].mean():.2f}  CTR={grp["ctr_r"].mean()*100:.2f}%  CPC={grp["depenses_fcfa"].sum()/grp["clics"].sum():.0f} FCFA')
print('\n✅ Ces valeurs sont vos références de validation dans Power BI')

> **MÉTHODE — Valider le dashboard avant de le livrer**
>
> Avant de partager un dashboard Power BI, vérifiez que les KPI cards correspondent aux valeurs de référence calculées ici. Si le ROAS affiché dans Power BI est différent du ROAS calculé en Python, c'est qu'une mesure DAX est incorrecte ou qu'un filtre involontaire est appliqué.
>
> **Checklist de validation :** ROAS global = **3.31×** · CTR = **2.86%** · CPC = **~127 FCFA** · Nb campagnes = **354** · Sous-perf = **26.3%** (93 campagnes)

In [ ]:
# Export des tables avec types corrects pour Power BI
# Power BI lit mieux les dates au format YYYY-MM-DD

# marketpulse_analytics — table centrale
df_an_export = df_an.copy()
for col in ['date_debut','date_fin']:
    if col in df_an_export.columns:
        df_an_export[col] = pd.to_datetime(df_an_export[col]).dt.strftime('%Y-%m-%d')
df_an_export.to_csv(f'{SAVE_PATH}pbi_marketpulse_analytics.csv', index=False)

# performances_daily — table des faits
df_perf_export = df_perf.copy()
df_perf_export['date_perf'] = pd.to_datetime(df_perf_export['date_perf']).dt.strftime('%Y-%m-%d')
df_perf_export['ctr_recalc']  = (df_perf_export['clics'] / df_perf_export['impressions'].replace(0, np.nan)).round(4)
df_perf_export['roas_recalc'] = np.where(
    (df_perf_export['revenu_genere_fcfa'] >= 0) & (df_perf_export['depenses_fcfa'] > 0),
    (df_perf_export['revenu_genere_fcfa'] / df_perf_export['depenses_fcfa']).round(3),
    np.nan
)
df_perf_export['is_clean'] = (
    (df_perf_export['clics'] <= df_perf_export['impressions']) &
    (df_perf_export['revenu_genere_fcfa'] >= 0) &
    (df_perf_export['roas'] <= 50) &
    (df_perf_export['ctr_recalc'] <= 0.20)
).astype(int)
df_perf_export.to_csv(f'{SAVE_PATH}pbi_performances_daily.csv', index=False)

print(f'✅ Fichiers Power BI exportés dans {SAVE_PATH}')
print(f'   pbi_marketpulse_analytics.csv : {len(df_an_export):,} lignes')
print(f'   pbi_performances_daily.csv    : {len(df_perf_export):,} lignes')
print(f'   (colonne is_clean ajoutée — utiliser comme filtre dans Power BI)')

---
## 1. Modèle de données Power BI

### MÉTHODE — Schéma en étoile
Le modèle MarketPulse est un **schéma en étoile** autour de `marketpulse_analytics` (table de faits campagne) et `performances_daily` (table de faits quotidienne). Les tables de dimension sont `clients_agence`, `audiences`, `contacts_crm` et `Calendrier`.

### Étapes de connexion dans Power BI Desktop

1. **Accueil → Obtenir des données → Texte/CSV**
2. Charger les 6 fichiers :
   - `pbi_marketpulse_analytics.csv`
   - `pbi_performances_daily.csv`
   - `clients_agence.csv`
   - `audiences.csv`
   - `contacts_crm.csv`
   - `campagnes_risque_scores.csv` *(créé en NB5 — charger en dernier)*
3. Pour chaque table CSV — dans **Power Query** :
   - Vérifier que les colonnes `date_*` sont détectées en type `Date`
   - S'assurer que les colonnes numériques (`roas_total`, `ctr_total`, etc.) sont en type `Nombre décimal`
   - Filtrer `pbi_performances_daily` sur `is_clean = 1` pour n'utiliser que les lignes propres

### Relations à créer (vue Modèle)

| Depuis | Vers | Clé | Cardinalité |
|---|---|---|---|
| `marketpulse_analytics[campagne_id]` | `pbi_performances_daily[campagne_id]` | `campagne_id` | 1 → N |
| `marketpulse_analytics[client_id]` | `clients_agence[client_id]` | `client_id` | N → 1 |
| `marketpulse_analytics[audience_id]` | `audiences[audience_id]` | `audience_id` | N → 1 |
| `marketpulse_analytics[client_id]` | `contacts_crm[client_id]` | `client_id` | 1 → N |
| `Calendrier[Date]` | `pbi_performances_daily[date_perf]` | `date` | 1 → N |
| `Calendrier[Date]` | `marketpulse_analytics[date_debut]` | `date` | 1 → N (inactive) |

> **Note :** La relation `Calendrier → marketpulse_analytics` sur `date_debut` est **inactive** car une table ne peut avoir qu'une seule relation active par colonne date. Pour filtrer par mois de lancement, utiliser `USERELATIONSHIP()` dans les mesures DAX.

---
## 2. Table Calendrier DAX

### MÉTHODE
La table Calendrier est **obligatoire** pour toutes les analyses temporelles (LAG mensuel, ROAS glissant 7j, filtres par période). On la crée en DAX depuis **Modélisation → Nouvelle table**.

### DAX — Table Calendrier

```dax
Calendrier =
VAR date_min = DATE(2022, 6, 1)
VAR date_max = DATE(2024, 5, 31)
RETURN
ADDCOLUMNS(
    CALENDAR(date_min, date_max),
    "Annee",           YEAR([Date]),
    "Mois",            MONTH([Date]),
    "Mois_Nom",        FORMAT([Date], "MMM YYYY"),
    "Mois_Abrege",     FORMAT([Date], "MMM"),
    "Trimestre",       "Q" & QUARTER([Date]),
    "Semaine",         WEEKNUM([Date]),
    "Jour_Semaine",    WEEKDAY([Date], 2),
    "Jour_Nom",        FORMAT([Date], "dddd"),
    "Est_Weekend",     IF(WEEKDAY([Date], 2) >= 6, 1, 0),
    "Annee_Mois",      FORMAT([Date], "YYYY-MM")
)
```

> **Trier `Mois_Nom` par `Mois` :** Dans la vue Données, sélectionner la colonne `Mois_Nom` → Outils de colonne → Trier par colonne → `Mois`. Cela garantit que les visuels affichent les mois dans l'ordre chronologique (Jan, Fév, Mar…) et non alphabétique.

---
## 3. Les 13 mesures DAX

### MÉTHODE
Créer une **table de mesures dédiée** (Modélisation → Entrer des données → table vide nommée `_Mesures`). Mettre toutes les mesures dans cette table — elle ne contient que des mesures, pas de données. Avantages :
- Visibilité : toutes les mesures en un endroit dans le volet Champs
- Maintenabilité : facilite les mises à jour
- Lisibilité : séparées des tables de données dans le volet

> **Convention de nommage :** préfixer par le périmètre — `[ROAS …]`, `[CTR …]`, `[ML …]`. Évite la confusion entre des mesures similaires.

### Mesure 1 — ROAS Global

```dax
ROAS Global =
DIVIDE(
    SUMX(
        FILTER(pbi_performances_daily, pbi_performances_daily[is_clean] = 1),
        pbi_performances_daily[revenu_genere_fcfa]
    ),
    SUMX(
        FILTER(pbi_performances_daily, pbi_performances_daily[is_clean] = 1),
        pbi_performances_daily[depenses_fcfa]
    ),
    0
)
```

> **Valeur attendue :** `3.31` — vérifier après création.

### Mesure 2 — ROAS 7 jours glissants

```dax
ROAS 7J Glissant =
VAR date_max = MAX(Calendrier[Date])
VAR date_debut = date_max - 6
RETURN
CALCULATE(
    DIVIDE(
        SUM(pbi_performances_daily[revenu_genere_fcfa]),
        SUM(pbi_performances_daily[depenses_fcfa]),
        0
    ),
    pbi_performances_daily[date_perf] >= date_debut,
    pbi_performances_daily[date_perf] <= date_max,
    pbi_performances_daily[is_clean] = 1
)
```

> **Usage :** Cette mesure change automatiquement selon la plage de dates sélectionnée dans les slicers. En l'absence de filtre, elle calcule les 7 derniers jours de la plage maximale disponible.

### Mesures 3 à 7 — CTR, CPC, CPA, Budget, Revenu

```dax
CTR Moyen % =
DIVIDE(
    SUMX(FILTER(pbi_performances_daily, [is_clean]=1), [clics]),
    SUMX(FILTER(pbi_performances_daily, [is_clean]=1), [impressions]),
    0
) * 100
```

```dax
CPC Moyen FCFA =
DIVIDE(
    SUMX(FILTER(pbi_performances_daily, [is_clean]=1), [depenses_fcfa]),
    SUMX(FILTER(pbi_performances_daily, [is_clean]=1), [clics]),
    0
)
```

```dax
CPA Moyen FCFA =
DIVIDE(
    SUMX(FILTER(pbi_performances_daily, [is_clean]=1), [depenses_fcfa]),
    SUMX(FILTER(pbi_performances_daily, [is_clean]=1), [conversions]),
    0
)
```

```dax
Budget Dépensé M FCFA =
DIVIDE(
    SUM(marketpulse_analytics[budget_depense_fcfa]),
    1000000
)
```

```dax
Revenu Total M FCFA =
DIVIDE(
    SUMX(
        FILTER(pbi_performances_daily, [is_clean]=1),
        [revenu_genere_fcfa]
    ),
    1000000
)
```

> **Valeurs attendues :** CTR = 2.86% · CPC = ~127 FCFA · CPA = ~7 287 FCFA

### Mesures 8 à 10 — Sous-performance

```dax
Nb Campagnes =
COUNTROWS(marketpulse_analytics)
```

```dax
Nb Campagnes Sous-Perf =
CALCULATE(
    COUNTROWS(marketpulse_analytics),
    marketpulse_analytics[campagne_sous_performe] = 1
)
```

```dax
% Campagnes Sous-Perf =
DIVIDE(
    [Nb Campagnes Sous-Perf],
    [Nb Campagnes],
    0
) * 100
```

> **Valeurs attendues :** Nb = 354 · Sous-perf = 93 · % = 26.3%

### Mesures 11 à 13 — LTV, ROAS J3 et Alertes ML

```dax
LTV Moyenne FCFA =
AVERAGEX(
    FILTER(contacts_crm, contacts_crm[nb_achats] > 0),
    contacts_crm[ltv_fcfa]
)
```

```dax
ROAS J3 Moyen =
AVERAGEX(
    FILTER(marketpulse_analytics, NOT(ISBLANK(marketpulse_analytics[roas_j3]))),
    marketpulse_analytics[roas_j3]
)
```

```dax
Nb Alertes ML Critiques =
CALCULATE(
    COUNTROWS(campagnes_risque_scores),
    campagnes_risque_scores[niveau_alerte] = "Critique"
)
```

```dax
Score ML Moyen =
AVERAGE(campagnes_risque_scores[score_risque])
```

> **Note :** Les mesures 12 et 13 requièrent `campagnes_risque_scores.csv` produit par le NB5. Créer des mesures placeholder retournant 0 pendant la phase de construction du dashboard, puis les mettre à jour une fois le NB5 complété.

### Mesure bonus — Alerte ROAS canal (pour formatage conditionnel)

```dax
Couleur ROAS Canal =
VAR roas_val = [ROAS Global]
RETURN
SWITCH(
    TRUE(),
    roas_val >= 4.0,  "#1D9E75",  -- vert DPL — excellent
    roas_val >= 2.5,  "#EF9F27",  -- orange — correct
    roas_val >= 1.5,  "#E24B4A",  -- rouge — sous seuil
                      "#791F1F"   -- rouge sombre — critique
)
```

> **Usage :** Cette mesure retourne un code couleur hexadécimal selon le niveau de ROAS. Elle s'utilise dans **Formatage conditionnel → Couleur d'arrière-plan → Valeur de champ** sur une carte ou une table.

---
## 4. Page 1 — Vue Executive

**Question business :** *Quelle est la santé globale du portefeuille campagnes ce mois-ci ?*

### CONFIGURATION

**En-tête de page**
- Fond : `#F5F5F8` (gris très clair — style EduTrack)
- Titre : `Vue executive — MarketPulse Analytics` en gras, couleur `#534AB7`
- Métadonnées à droite : plage de dates active (slicer date)

**Bandeau alerte (haut de page)**
- Visuel : **Carte** ou **Zone de texte** avec fond `#FEF2F2`, bordure `#E24B4A`
- Formule DAX pour le texte dynamique :

```dax
Texte Alerte =
VAR nb_crit = [Nb Alertes ML Critiques]
VAR pct_sp  = ROUND([% Campagnes Sous-Perf], 1)
RETURN
IF(
    nb_crit > 0,
    "⚠  " & nb_crit & " campagne(s) en alerte critique  |  "
        & pct_sp & "% de sous-performance globale",
    "✓  Aucune alerte critique — ROAS global : " & ROUND([ROAS Global], 2) & "×"
)
```

**4 KPI cards (ligne horizontale)**

| Card | Mesure | Format | Couleur accent |
|---|---|---|---|
| Nb campagnes | `[Nb Campagnes]` | Entier | `#534AB7` violet |
| ROAS global | `[ROAS Global]` | `0.00"×"` | Variable (`[Couleur ROAS Canal]`) |
| CTR moyen | `[CTR Moyen %]` | `0.00"%"` | `#EF9F27` ambre |
| % sous-perf | `[% Campagnes Sous-Perf]` | `0.0"%"` | `#E24B4A` rouge |

> **Formatage conditionnel des KPI cards :** Sélectionner le visuel Card → Format → Valeur de légende → Couleur de police → Valeur de champ → mesure `[Couleur ROAS Canal]`. Le chiffre ROAS passe automatiquement au vert si > 4.0, orange si 2.5-4.0, rouge si < 2.5.

**Graphique principal (centre-gauche)**
- Visuel : **Graphique à barres groupées + courbe** (combiné)
- Axe X : `Calendrier[Mois_Nom]` (trié par `Mois`)
- Barres : `SUMX(pbi_performances_daily, [depenses_fcfa])` — couleur `#534AB7`
- Courbe : `[ROAS 7J Glissant]` — couleur `#E24B4A`, axe secondaire
- Titre : `Dépenses mensuelles & ROAS glissant 7J`
- Ligne de référence Y2 à 1.5 (seuil sous-performance) — couleur `#E24B4A` pointillé

**Panel top corridors (droite)**
- Visuel : **Tableau** ou **Graphique à barres horizontales**
- Colonnes : `canal`, `[ROAS Global]`, `[CTR Moyen %]`, `[% Campagnes Sous-Perf]`
- Tri : par `[ROAS Global]` décroissant
- Formatage conditionnel barres de données sur ROAS — palette Vert → Rouge

**Slicers**
- Slicer 1 : `Calendrier[Annee_Mois]` — type liste déroulante
- Slicer 2 : `clients_agence[nom]` — type liste avec recherche
- Slicer 3 : `marketpulse_analytics[canal]` — type boutons

**Navigation entre pages**
- Ajouter 5 boutons en bas de page liés à chaque page (`Insertion → Bouton → Navigation de page`)

---
## 5. Page 2 — Performance des Canaux

**Question business :** *Sur quel canal investir le budget du mois prochain ?*

### CONFIGURATION

**Mesures DAX supplémentaires pour cette page :**

```dax
ROAS Par Canal =
AVERAGEX(
    marketpulse_analytics,
    marketpulse_analytics[roas_total]
)
```

```dax
Rang ROAS Canal =
RANKX(
    ALL(marketpulse_analytics[canal]),
    [ROAS Par Canal],
    ,
    DESC,
    DENSE
)
```

**Visuel 1 — Graphique en anneau (top-left)**
- Part du budget par canal : `SUM(marketpulse_analytics[budget_depense_fcfa])`
- Légende : `marketpulse_analytics[canal]`
- Palette : Email `#1D9E75` · SMS `#534AB7` · Google `#EF9F27` · Facebook `#E24B4A`

**Visuel 2 — Tableau comparatif canaux (centre)**
- Colonnes : `canal`, `[Nb Campagnes]`, `[ROAS Global]`, `[CTR Moyen %]`, `[CPC Moyen FCFA]`, `[CPA Moyen FCFA]`, `[% Campagnes Sous-Perf]`, `[Rang ROAS Canal]`
- Tri par rang ROAS
- Barres de données conditionnelles sur `[ROAS Global]` : palette Vert DPL `#1D9E75` → Rouge `#E24B4A`
- Ligne de total désactivée (le total d'un classement n'a pas de sens)

**Visuel 3 — Heatmap jour × canal (bas de page)**
- Importer depuis le PNG généré en NB3 : `Insertion → Image`
- Ou recréer en Power BI : Matrice avec `canal` en lignes, `Calendrier[Jour_Nom]` en colonnes, `[ROAS Global]` en valeurs
- Formatage conditionnel : fond avec dégradé Vert → Blanc → Rouge selon la valeur

**Visuel 4 — Tendance ROAS mensuelle par canal (bas droite)**
- Graphique en courbes
- Axe X : `Calendrier[Mois_Nom]`
- Légende : `marketpulse_analytics[canal]`
- Valeur : `[ROAS Par Canal]`
- Ligne de référence Y à 1.5 (seuil)

```dax
ROAS Mensuel Canal =
CALCULATE(
    AVERAGEX(marketpulse_analytics, [roas_total]),
    USERELATIONSHIP(Calendrier[Date], marketpulse_analytics[date_debut])
)
```

> **Note `USERELATIONSHIP` :** La relation `Calendrier → marketpulse_analytics` sur `date_debut` est inactive dans le modèle (pour éviter l'ambiguïté avec `performances_daily`). `USERELATIONSHIP` l'active localement dans cette mesure uniquement.

---
## 6. Page 3 — Audiences & Créatifs

**Question business :** *Quelle audience et quel type de créatif performent le mieux ?*

### CONFIGURATION

**Mesures DAX supplémentaires :**

```dax
Indice Saturation Moyen =
AVERAGEX(
    FILTER(marketpulse_analytics,
           NOT(ISBLANK(marketpulse_analytics[indice_saturation_j3]))),
    marketpulse_analytics[indice_saturation_j3]
)
```

```dax
ROAS Par Type Audience =
AVERAGEX(
    marketpulse_analytics,
    marketpulse_analytics[roas_total]
)
```

**Visuel 1 — Tableau type audience × canal**
- Lignes : `audiences[type_audience]` (Froide / Lookalike / Retargeting / CRM)
- Colonnes : `marketpulse_analytics[canal]`
- Valeur : `[ROAS Par Type Audience]`
- Visuellement : Matrice Power BI avec barres de données par cellule

**Visuel 2 — ROAS par tranche d'âge**
- Graphique à barres horizontales
- Axe Y : `audiences[tranche_age]`
- Valeur X : `[ROAS Par Type Audience]`
- Couleur par barre selon `[Couleur ROAS Canal]`

**Visuel 3 — Saturation audience (scatter plot)**
- Graphique à nuage de points
- Axe X : `[Indice Saturation Moyen]`
- Axe Y : `[ROAS Global]`
- Taille bulle : `[Nb Campagnes]`
- Légende couleur : `audiences[type_audience]`
- Ce visuel révèle le **point de saturation** : au-delà d'un certain indice, le ROAS chute

**Visuel 4 — Performance taux CTR J3 par type**
- KPI card : `AVERAGEX(marketpulse_analytics, [ctr_j3])` — CTR moyen en J3
- Comparaison vs CTR total (`[CTR Moyen %]`)
- Delta = `[CTR Moyen %] - AVERAGEX(marketpulse_analytics, [ctr_j3] * 100)` → si positif, les campagnes s'améliorent après J3

> **Insight à mettre en avant :** Les audiences **CRM** ont systématiquement le meilleur ROAS mais le plus petit volume. Les audiences **Froide** ont le plus grand volume mais le ROAS le plus bas. Ce trade-off entre échelle et performance est central dans la stratégie de priorisation budgétaire.

---
## 7. Page 4 — Suivi des Clients Agence

**Question business :** *Quels clients sont en risque de nous quitter ? Qui performe le mieux ?*

### CONFIGURATION

**Mesures DAX supplémentaires :**

```dax
ROAS Client Delta Mensuel =
VAR roas_actuel =
    CALCULATE(
        AVERAGEX(marketpulse_analytics, [roas_total]),
        USERELATIONSHIP(Calendrier[Date], marketpulse_analytics[date_debut])
    )
VAR roas_precedent =
    CALCULATE(
        AVERAGEX(marketpulse_analytics, [roas_total]),
        DATEADD(Calendrier[Date], -1, MONTH),
        USERELATIONSHIP(Calendrier[Date], marketpulse_analytics[date_debut])
    )
RETURN roas_actuel - roas_precedent
```

```dax
Statut Client =
VAR delta = [ROAS Client Delta Mensuel]
VAR roas  = [ROAS Par Canal]
RETURN
SWITCH(
    TRUE(),
    delta < -0.10 && roas < 2.0, "🔴 Critique",
    delta < -0.05,               "🟠 Élevé",
    delta < 0,                   "🟡 Modéré",
                                 "🟢 Stable"
)
```

**Visuel 1 — Scatter matrice risque clients**
- Graphique à nuage de points
- Axe X : `[ROAS Global]` (performance actuelle)
- Axe Y : `[ROAS Client Delta Mensuel]` (tendance)
- Taille bulle : `[Budget Dépensé M FCFA]`
- Couleur : `clients_agence[secteur]`
- Ligne de référence X à 2.0 (seuil risque) et Y à 0 (bascule hausse/baisse)
- Ce quadrant identifie visuellement les 4 profils : Croissance / À surveiller / En danger / Stable

**Visuel 2 — Tableau clients avec indicateurs**
- Colonnes : `clients_agence[nom]`, `ca[secteur]`, `[Nb Campagnes]`, `[ROAS Global]`, `[% Campagnes Sous-Perf]`, `[ROAS Client Delta Mensuel]`, `[Statut Client]`
- Tri par `[% Campagnes Sous-Perf]` décroissant
- Formatage conditionnel colonne `[Statut Client]` : couleur de fond selon valeur texte

**Configuration formatage conditionnel avancé :**
```
Colonne [Statut Client] → Format → Formatage conditionnel → Couleur d'arrière-plan
Règles :
  Valeur contient "Critique" → fond #FCEBEB, texte #E24B4A
  Valeur contient "Élevé"   → fond #FAEEDA, texte #EF9F27
  Valeur contient "Modéré"  → fond #FFFEF0, texte #BA7517
  Valeur contient "Stable"  → fond #E1F5EE, texte #0F6E56
```

**Visuel 3 — Courbes tendance ROAS par client (top 5 et bottom 5)**
- Filtrage Top N : Slicer → Filtres visuels → Top N → 5 par `[ROAS Global]`
- Courbes avec légende `clients_agence[nom]`
- Axe X : `Calendrier[Mois_Nom]`

**Visuel 4 — KPIs LTV par canal**
- Graphique à barres
- Axe : `contacts_crm[canal_acquisition]`
- Valeurs : `[LTV Moyenne FCFA]`, `[Nb Campagnes]` (axe secondaire)
- Titre : `LTV moyenne par canal d'acquisition`

---
## 8. Page 5 — Alertes ML · Prédiction de sous-performance

**Question business :** *Quelles campagnes actives vont sous-performer — et que faire dès maintenant ?*

> **Prérequis :** `campagnes_risque_scores.csv` doit être produit par le NB5 et chargé dans Power BI. Ce fichier contient une ligne par campagne avec `score_risque`, `niveau_alerte` (Critique / Élevé / Modéré) et `action_recommandee`.

### CONFIGURATION

**Bandeau critique (haut de page — formatage conditionnel dynamique)**
- Visuel : Rectangle coloré `#FEF2F2`, bordure `#E24B4A`
- Texte : mesure `[Texte Alerte]`
- Ce bandeau devient vert si aucune alerte critique n'est active

**3 cartes niveaux (ligne horizontale)**

```dax
Nb Alertes Élevées =
CALCULATE(
    COUNTROWS(campagnes_risque_scores),
    campagnes_risque_scores[niveau_alerte] = "Élevé"
)

Nb Alertes Modérées =
CALCULATE(
    COUNTROWS(campagnes_risque_scores),
    campagnes_risque_scores[niveau_alerte] = "Modéré"
)
```

| Card | Mesure | Couleur fond | Couleur texte |
|---|---|---|---|
| Alertes Critiques | `[Nb Alertes ML Critiques]` | `#FCEBEB` | `#E24B4A` |
| Alertes Élevées | `[Nb Alertes Élevées]` | `#FAEEDA` | `#EF9F27` |
| Alertes Modérées | `[Nb Alertes Modérées]` | `#EEEDFE` | `#534AB7` |

**Tableau prioritaire des alertes**
- Visuels : **Tableau** Power BI
- Source : table `campagnes_risque_scores` jointe à `clients_agence` et `marketpulse_analytics`
- Colonnes :

```
campagnes_risque_scores[campagne_id]
clients_agence[nom]           → client
marketpulse_analytics[canal]
marketpulse_analytics[objectif_campagne]
marketpulse_analytics[date_debut]
campagnes_risque_scores[score_risque]
campagnes_risque_scores[niveau_alerte]
campagnes_risque_scores[action_recommandee]
```

- Tri : `score_risque` décroissant
- Formatage conditionnel colonne `score_risque` : barres de données rouges 0 → 1
- Formatage conditionnel colonne `niveau_alerte` : couleur fond selon valeur (même règles que Page 4)

**Graphique donut — Répartition des alertes**
- Légende : `campagnes_risque_scores[niveau_alerte]`
- Valeur : `COUNTROWS(campagnes_risque_scores)`
- Couleurs : Critique `#E24B4A` · Élevé `#EF9F27` · Modéré `#888780`

**Fiche modèle ML (bas de page)**
- Zone de texte avec informations du modèle :
```
Modèle : Random Forest
Recall : 0.96 | AUC : 0.86 | Seuil : 0.30
Entraîné sur campagnes jusqu'au 30/06/2023
Dernière mise à jour : [date NB5]
Recommandation : ré-entraînement trimestriel
```

> **Bandeau rouge conditionnel — Configuration complète :**
> 1. Créer un Rectangle → Remplissage : valeur de champ → mesure `[Couleur Fond Alerte]`
> 2. `[Couleur Fond Alerte]` = `IF([Nb Alertes ML Critiques] > 0, "#FEF2F2", "#E1F5EE")`
> 3. Superposer une Zone de texte avec mesure `[Texte Alerte]` — police rouge ou verte selon `[Nb Alertes ML Critiques]`
> 4. Lier les deux visuels à `campagnes_risque_scores` pour qu'ils se mettent à jour automatiquement à chaque rechargement

---
## 9. Checklist finale avant livraison

### Validation des KPIs

Vérifier dans le dashboard sans aucun filtre actif :

```
☐  ROAS Global          = 3.31×
☐  CTR Moyen %          = 2.86%
☐  CPC Moyen FCFA       ≈ 127
☐  Nb Campagnes         = 354
☐  % Sous-Performance   = 26.3% (93 campagnes)
☐  Nb alertes Email     = 0  (Email n'a jamais sous-performé)
☐  ROAS Email           ≈ 4.67×
☐  ROAS Facebook        ≈ 2.51×
☐  LTV Moyenne          ≈ voir NB3
```

### Validation des interactions

```
☐  Slicer canal = Email → les 4 KPI cards se mettent à jour
☐  Clic sur barre mensuelle → filtre les autres visuels de la page
☐  Navigation entre pages (5 boutons) fonctionnelle
☐  Formatage conditionnel rouge sur ROAS < 1.5
☐  Tableau alertes ML trié par score décroissant
☐  ROAS 7J Glissant varie selon la plage de dates sélectionnée
```

### Paramètres de publication

```
☐  Nom du fichier      : MarketPulse_Dashboard_v1.pbix
☐  Onglet Page 1       : actif par défaut à l'ouverture
☐  Thème              : Importer le JSON ci-dessous
☐  Actualisation       : Planifier actualisation quotidienne (Power BI Service)
```

### Thème Power BI JSON (palette DataProjectLab)

```json
{
  "name": "DataProjectLab",
  "dataColors": [
    "#534AB7",
    "#1D9E75",
    "#EF9F27",
    "#E24B4A",
    "#888780",
    "#3B82F6",
    "#AFA9EC",
    "#5DCAA5"
  ],
  "background": "#F5F5F8",
  "foreground": "#1E1E3F",
  "tableAccent": "#534AB7"
}
```

> **Importer le thème :** Affichage → Thèmes → Parcourir les thèmes → sélectionner le fichier `.json` — toutes les couleurs des visuels s'alignent automatiquement sur la palette DataProjectLab.

In [ ]:
# Vérification automatique des KPIs de référence
import duckdb

conn_check = duckdb.connect()
conn_check.execute(f"CREATE TABLE an AS SELECT * FROM read_csv_auto('{analytics_path if os.path.exists(analytics_path) else BASE_URL+\"marketpulse_analytics.csv\"}')")
conn_check.execute(f"CREATE TABLE pd AS SELECT * FROM read_csv_auto('{BASE_URL}performances_daily.csv')")

res = conn_check.execute("""
    WITH clean AS (
        SELECT * FROM pd
        WHERE clics <= impressions
          AND revenu_genere_fcfa >= 0
          AND roas <= 50
          AND clics * 1.0 / NULLIF(impressions, 0) <= 0.20
    )
    SELECT
        ROUND(SUM(revenu_genere_fcfa) * 1.0 / NULLIF(SUM(depenses_fcfa), 0), 3) AS roas_global,
        ROUND(SUM(clics) * 100.0 / NULLIF(SUM(impressions), 0), 2)              AS ctr_pct,
        ROUND(SUM(depenses_fcfa) / NULLIF(SUM(clics), 0), 0)                    AS cpc_fcfa
    FROM clean
""").df()

nb_camp = conn_check.execute("SELECT COUNT(*), SUM(campagne_sous_performe) FROM an").df()

print('=== CHECKLIST AUTOMATIQUE — VALEURS À RETROUVER DANS POWER BI ===')
print(f'  ROAS Global          : {res["roas_global"].iloc[0]:.3f}×  → attendu : 3.31×')
print(f'  CTR Moyen %          : {res["ctr_pct"].iloc[0]:.2f}%  → attendu : 2.86%')
print(f'  CPC Moyen FCFA       : {int(res["cpc_fcfa"].iloc[0])}  → attendu : ~127 FCFA')
print(f'  Nb Campagnes         : {nb_camp.iloc[0,0]}  → attendu : 354')
print(f'  Nb Sous-Perf         : {nb_camp.iloc[0,1]}  → attendu : 93')
print(f'  % Sous-Perf          : {nb_camp.iloc[0,1]/nb_camp.iloc[0,0]*100:.1f}%  → attendu : 26.3%')
print('\n✅ Ces valeurs sont vos références de validation dans Power BI')

---
## Bilan du Notebook 4

### Récapitulatif du dashboard

| Page | Question business | Visuels clés |
|---|---|---|
| **1 — Vue Executive** | Santé globale du portefeuille | KPI cards + barres mensuelles + bandeau alerte |
| **2 — Performance Canaux** | Sur quel canal investir ? | Anneau budget + tableau comparatif + heatmap |
| **3 — Audiences & Créatifs** | Quelle audience performe ? | Matrice type×canal + scatter saturation |
| **4 — Suivi Clients** | Qui va nous quitter ? | Scatter risque + tableau `[Statut Client]` coloré |
| **5 — Alertes ML** | Quelles campagnes stopper dès maintenant ? | Bandeau rouge + tableau scores + donut |

### 13 mesures DAX créées

| # | Mesure | Usage |
|---|---|---|
| 1 | `ROAS Global` | KPI card Page 1 |
| 2 | `ROAS 7J Glissant` | Courbe Page 1 + 2 |
| 3 | `CTR Moyen %` | KPI card Pages 1 et 2 |
| 4 | `CPC Moyen FCFA` | Tableau Page 2 |
| 5 | `CPA Moyen FCFA` | Tableau Pages 2 et 4 |
| 6 | `Budget Dépensé M FCFA` | KPI card Pages 1 et 4 |
| 7 | `Revenu Total M FCFA` | KPI card Page 1 |
| 8 | `Nb Campagnes` | KPI card toutes pages |
| 9 | `Nb Campagnes Sous-Perf` | Page 1 et 5 |
| 10 | `% Campagnes Sous-Perf` | KPI card Pages 1 et 4 |
| 11 | `LTV Moyenne FCFA` | Page 4 |
| 12 | `ROAS J3 Moyen` | Page 5 |
| 13 | `Nb Alertes ML Critiques` | Bandeau Pages 1 et 5 |

**Pour le NB5 :** entraîner le modèle Random Forest avec coupure temporelle, optimiser le seuil Recall ≥ 0.75, exporter `campagnes_risque_scores.csv` avec les colonnes `campagne_id`, `score_risque`, `niveau_alerte`, `action_recommandee`.

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.